# Benchmark: Flow-Sinkhorn vs Vanilla Sinkhorn on WOT Graphs

This notebook benchmarks:

- `flow_sinkhorn_sparse` on k-NN graph edges.
- `vanilla_sinkhorn_dense` on all-pairs graph shortest-path metric.
- `lp_highs` exact unregularized graph W1 reference.

It reports separately:

1. Floyd-Warshall precompute overhead (for dense metric baseline).
2. Solver runtimes.
3. Accuracy vs LP reference (relative objective error).
4. Dual-potential error (`L_inf` after centering).
5. Constraint residual (`L1` divergence error).

We vary graph size through `n0` (cells/timepoint) and vary `epsilon`.


In [ ]:
# Optional first-time setup
# !pip install -e .
# !pip install -r singlecell/requirements.txt


In [ ]:
# Compute backend configuration
USE_GPU = False  # set True to use GPU when available
DEVICE = 'cuda' if USE_GPU else 'cpu'
USE_TORCH = True  # torch sparse solver backend


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from benchmarks.wot_benchmark import (
    BenchmarkConfig,
    prepare_wot_data,
    benchmark_methods_on_prepared_data,
)


In [ ]:
DATA_ROOT = Path("data/wot")

n0_values = [50, 100, 150, 200]
eps_values = [0.2, 0.1, 0.05, 0.02]
max_timepoints = 8   # keeps dense baseline feasible
niter = 300

all_rows = []
for n0 in n0_values:
    cfg = BenchmarkConfig(
        data_root=DATA_ROOT,
        random_state=0,
        n_cells_per_time=n0,
        pca_components=30,
        knn_k=4,
        niter=niter,
        use_torch=USE_TORCH,
        max_timepoints=max_timepoints,
    )
    prepared = prepare_wot_data(cfg)
    df = benchmark_methods_on_prepared_data(
        prepared,
        epsilons=eps_values,
        niter=niter,
        use_torch=USE_TORCH,
        device=DEVICE,
    )
    df["n0"] = n0
    df["n_days"] = len(prepared["days"])
    all_rows.append(df)

bench = pd.concat(all_rows, ignore_index=True)
bench.head(20)


In [ ]:
# Split tables for readability
fw_lp = bench[bench["method"].isin(["floyd_warshall", "lp_highs"])].copy()
solvers = bench[bench["method"].isin(["flow_sinkhorn_sparse", "vanilla_sinkhorn_dense"])].copy()

fw_lp


In [ ]:
# Precompute overhead: Floyd-Warshall vs n0
fw = fw_lp[fw_lp["method"] == "floyd_warshall"].sort_values("n0")
plt.figure(figsize=(6, 3.5))
plt.plot(fw["n0"], fw["time_sec"], marker="o", label="Floyd-Warshall")
plt.xlabel("n0 (cells per timepoint)")
plt.ylabel("time (sec)")
plt.title("All-pairs shortest-path precompute overhead")
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Time vs accuracy (relative objective error)
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for method, ax in zip(["flow_sinkhorn_sparse", "vanilla_sinkhorn_dense"], axes):
    sub = solvers[solvers["method"] == method]
    for n0 in sorted(sub["n0"].unique()):
        ss = sub[sub["n0"] == n0].sort_values("objective_rel_err_vs_lp")
        ax.plot(ss["objective_rel_err_vs_lp"], ss["time_sec"], marker="o", label=f"n0={n0}")
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel("relative objective error vs LP")
    ax.set_title(method)
    ax.grid(alpha=0.25)
axes[0].set_ylabel("time (sec)")
axes[1].legend(title="graph size", fontsize=8)
plt.tight_layout()
plt.show()


In [ ]:
# Dual L_inf error vs epsilon
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for method, ax in zip(["flow_sinkhorn_sparse", "vanilla_sinkhorn_dense"], axes):
    sub = solvers[solvers["method"] == method]
    for n0 in sorted(sub["n0"].unique()):
        ss = sub[sub["n0"] == n0].sort_values("epsilon", ascending=False)
        ax.plot(ss["epsilon"], ss["dual_linf_vs_lp"], marker="o", label=f"n0={n0}")
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel("epsilon")
    ax.set_title(method)
    ax.grid(alpha=0.25)
axes[0].set_ylabel("dual L_inf error vs LP")
axes[1].legend(title="graph size", fontsize=8)
plt.tight_layout()
plt.show()


In [ ]:
# Constraint residuals
pivot = solvers.pivot_table(
    index=["n0", "epsilon"],
    columns="method",
    values=["constraint_l1", "time_sec", "objective_rel_err_vs_lp", "dual_linf_vs_lp"],
)
pivot


In [ ]:
# Save benchmark table
out = Path("singlecell/results/benchmark_flowsinkhorn_vs_vanilla.csv")
out.parent.mkdir(parents=True, exist_ok=True)
bench.to_csv(out, index=False)
out
